[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_50_Track5_Capstone_ProductionAIStack.ipynb)

# Lesson 50 -- Track 5 Capstone: Production AI Observability Stack

## Track 5 Retrospective

| Lesson | Topic | Core Concept |
|--------|-------|--------------|
| L46 | Durable Execution | SQLite step-cache, crash-safe pipelines |
| L47 | GPU Autoscaling | vLLM queue depth -> HPA, scale-to-zero |
| L48 | OTel Tracing | Per-span breakdown, flamegraphs |
| L49 | Eval at Scale | Async judge queue, regression detector |
| **L50** | **Capstone** | **Wire all four into one production stack** |

## Capstone Goal

Build `ProductionAIStack` -- a single class where:

- Every request runs through a **durable pipeline** (step-cached, crash-safe)
- Every request is **traced end-to-end** (root -> durable -> llm spans)
- **Prometheus-format metrics** are exported at `/metrics` (requests, latency, eval scores)
- A sample of every response is **judged async** -- zero hot-path blocking
- **Regression alerts** fire when eval quality drops statistically
- The whole thing ships as **`docker compose up`** with Grafana pre-wired

By the end of this lesson you have a deployable repo skeleton.


## System Architecture

```
REQUEST
   |
   v
+-----------------------------------------------------------+
|                  ProductionAIStack                        |
|                                                           |
|  +---------------+   +--------------+  +--------------+  |
|  | Durable       |   | OTel         |  | Prometheus   |  |
|  | Pipeline      |-->| Tracer       |  | Metrics      |  |
|  | (L46)         |   | (L48)        |  | /metrics     |  |
|  +-------+-------+   +--------------+  +--------------+  |
|          | response                                       |
|          v                                                |
|  +---------------+   +--------------+  +--------------+  |
|  | Importance    |   | Async Eval   |  | Eval Metric  |  |
|  | Sampler       |-->| Pipeline     |->| Store        |  |
|  | (L49)         |   | (L49)        |  | (SQLite)     |  |
|  +---------------+   +--------------+  +------+-------+  |
|                                               |           |
|                                    +----------v-------+   |
|                                    | Regression       |   |
|                                    | Detector         |   |
|                                    | (Mann-Whitney U) |   |
|                                    +------------------+   |
+-----------------------------------------------------------+

Production deployment (docker-compose):

  +-------------+  OTLP gRPC  +----------+
  |  AI App     |------------>|  Jaeger  |
  |  :8000      |             |  :16686  |
  +------+------+             +----------+
         | /metrics scrape
         v
  +-------------+  datasource  +----------+
  | Prometheus  |------------->|  Grafana |
  |   :9090     |              |  :3000   |
  +-------------+              +----------+
```

**Three design rules:**
1. The hot path is: `durable_research -> OTel spans -> metrics.record()`. Eval is always off-path.
2. One `infer()` call produces all four signals simultaneously.
3. The same `ProductionAIStack` works in Colab (in-memory exporters) and production (OTLP exporters).


In [ ]:
# Setup -- install all Track 5 dependencies
!pip install anthropic nest_asyncio pandas matplotlib scipy \
    opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-grpc \
    pyyaml -q

print('All dependencies installed')


In [ ]:
import asyncio, json, os, random, sqlite3, time, uuid
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable

import nest_asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import mannwhitneyu

import anthropic
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import StatusCode

nest_asyncio.apply()

# Anthropic client
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    api_key = os.environ.get('ANTHROPIC_API_KEY', 'your-key-here')

client       = anthropic.Anthropic(api_key=api_key)
async_client = anthropic.AsyncAnthropic(api_key=api_key)

HAIKU  = 'claude-haiku-4-5'
SONNET = 'claude-sonnet-4-5'

# OTel: in-memory exporter for Colab
# Production swap: OTLPSpanExporter('http://jaeger:4317') + BatchSpanProcessor
_span_exporter   = InMemorySpanExporter()
_tracer_provider = TracerProvider()
_tracer_provider.add_span_processor(SimpleSpanProcessor(_span_exporter))
trace.set_tracer_provider(_tracer_provider)
TRACER = trace.get_tracer('ai.production.stack')

print(f'Ready | HAIKU={HAIKU}')
print('OTel TracerProvider configured with InMemorySpanExporter')


## Component 1 -- Durable Pipeline (L46 Core)

The durable pipeline gives us **crash-safe, step-cached** LLM calls.
Any step already completed is replayed from SQLite -- no re-billing, no extra latency.

**Key design:** `DurableCtx.run(name, fn, *args)` is the only place LLM calls live.
`WorkflowDB` is the boundary between *done* and *not yet done*.

In Colab: `WorkflowDB(':memory:')` -- resets on kernel restart.
In production: `WorkflowDB('/data/workflows.db')` -- survives crashes.


In [ ]:
# Component 1: Durable Pipeline

class WorkflowDB:
    def __init__(self, path=':memory:'):
        self.conn = sqlite3.connect(path, check_same_thread=False)
        self.conn.execute(
            'CREATE TABLE IF NOT EXISTS steps '
            '(wf_id TEXT, step TEXT, result TEXT, ts REAL, '
            ' PRIMARY KEY (wf_id, step))')
        self.conn.commit()

    def get(self, wf_id, step):
        row = self.conn.execute(
            'SELECT result FROM steps WHERE wf_id=? AND step=?',
            (wf_id, step)).fetchone()
        return json.loads(row[0]) if row else None

    def put(self, wf_id, step, result):
        self.conn.execute(
            'INSERT OR REPLACE INTO steps VALUES (?,?,?,?)',
            (wf_id, step, json.dumps(result), time.time()))
        self.conn.commit()


@dataclass
class DurableCtx:
    wf_id: str
    db: WorkflowDB
    _counts: dict = field(default_factory=dict)
    replayed: int = 0
    executed: int = 0

    def run(self, name: str, fn: Callable, *args, **kwargs):
        n = self._counts.get(name, 0)
        self._counts[name] = n + 1
        key = f'{name}:{n}'
        cached = self.db.get(self.wf_id, key)
        if cached is not None:
            self.replayed += 1
            return cached
        result = fn(*args, **kwargs)
        self.db.put(self.wf_id, key, result)
        self.executed += 1
        return result


WF_DB = WorkflowDB()


def _llm(system: str, user: str, model=HAIKU, max_tokens=300) -> str:
    r = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system, messages=[{'role': 'user', 'content': user}])
    return r.content[0].text


def durable_research(wf_id: str, query: str) -> dict:
    ctx = DurableCtx(wf_id=wf_id, db=WF_DB)
    questions = ctx.run('expand', _llm,
        'Expand this query into 2 focused sub-questions (numbered list).', query)
    draft = ctx.run('draft', _llm,
        'Answer these questions concisely in 3-4 sentences.', questions)
    answer = ctx.run('revise', _llm,
        'Improve this draft: fix any inaccuracies, add one concrete example.', draft)
    return {'answer': answer, 'replayed': ctx.replayed, 'executed': ctx.executed}


# Sanity check
_r = durable_research(f'wf-{uuid.uuid4()}', 'What is attention in transformers?')
print(f'Durable pipeline OK | executed={_r["executed"]} replayed={_r["replayed"]}')
print(f'Answer preview: {_r["answer"][:120]}...')


In [ ]:
# Component 2: OTel Tracing
# Wraps durable_research in a two-level span hierarchy:
#   pipeline.request (root)
#     pipeline.durable (child -- all LLM steps run inside)

def traced_infer(query: str, tag: str, wf_id: str) -> dict:
    with TRACER.start_as_current_span('pipeline.request') as root:
        root.set_attribute('req.query',  query[:80])
        root.set_attribute('req.tag',    tag)
        root.set_attribute('req.wf_id',  wf_id)
        t0 = time.perf_counter()

        try:
            with TRACER.start_as_current_span('pipeline.durable') as child:
                result = durable_research(wf_id, query)
                child.set_attribute('durable.replayed', result['replayed'])
                child.set_attribute('durable.executed', result['executed'])

            latency = time.perf_counter() - t0
            root.set_attribute('req.latency_s',  round(latency, 3))
            root.set_attribute('req.answer_len', len(result['answer']))
            root.set_status(StatusCode.OK)
            return {**result, 'latency_s': latency, 'tag': tag, 'wf_id': wf_id}

        except Exception as e:
            root.record_exception(e)
            root.set_status(StatusCode.ERROR, str(e))
            raise


# Smoke test
_r2 = traced_infer('How does BERT differ from GPT?', tag='nlp', wf_id=f'wf-{uuid.uuid4()}')
_spans = _span_exporter.get_finished_spans()
print(f'OTel tracing OK | {len(_spans)} spans captured so far')
for s in _spans[-2:]:
    dur_ms = (s.end_time - s.start_time) / 1_000_000
    print(f'  {s.name:35s}  {s.status.status_code.name}  {dur_ms:.0f}ms')

# EXPERIMENT: swap for OTLP in production:
# from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
# from opentelemetry.sdk.trace.export import BatchSpanProcessor
# _tracer_provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter('http://jaeger:4317')))


In [ ]:
# Component 3: Prometheus-Exportable Metrics

@dataclass
class ProdMetrics:
    _req_count:   Counter = field(default_factory=Counter)
    _latencies:   list    = field(default_factory=list)
    _errors:      int     = 0
    _eval_scores: list    = field(default_factory=list)

    def record(self, tag: str, latency_s: float, ok: bool):
        self._req_count[tag] += 1
        self._latencies.append(latency_s)
        if not ok:
            self._errors += 1

    def record_score(self, score: float):
        self._eval_scores.append(score)

    @property
    def p95_latency(self) -> float:
        return float(np.percentile(self._latencies, 95)) if self._latencies else 0.0

    @property
    def mean_score(self) -> float:
        return float(np.mean(self._eval_scores)) if self._eval_scores else 0.0

    @property
    def poor_rate(self) -> float:
        if not self._eval_scores: return 0.0
        return sum(s < 0.6 for s in self._eval_scores) / len(self._eval_scores)

    def prometheus(self) -> str:
        lines = [
            '# HELP ai_requests_total Total inference requests by tag',
            '# TYPE ai_requests_total counter',
        ]
        for tag, count in self._req_count.items():
            lines.append(f'ai_requests_total{{tag="{tag}"}} {count}')
        lines += [
            '# HELP ai_latency_p95_seconds P95 latency in seconds',
            '# TYPE ai_latency_p95_seconds gauge',
            f'ai_latency_p95_seconds {self.p95_latency:.4f}',
            '# HELP ai_eval_mean_score Rolling mean judge score (0-1)',
            '# TYPE ai_eval_mean_score gauge',
            f'ai_eval_mean_score {self.mean_score:.4f}',
            '# HELP ai_eval_poor_rate Fraction of responses scoring below 0.6',
            '# TYPE ai_eval_poor_rate gauge',
            f'ai_eval_poor_rate {self.poor_rate:.4f}',
            '# HELP ai_errors_total Total errors',
            '# TYPE ai_errors_total counter',
            f'ai_errors_total {self._errors}',
        ]
        return '\n'.join(lines)


METRICS = ProdMetrics()
print('Prometheus metrics ready. Sample /metrics output with dummy data:')
_demo = ProdMetrics()
_demo.record('nlp', 2.1, True); _demo.record('ml', 1.8, True)
_demo.record_score(0.82); _demo.record_score(0.55)
print(_demo.prometheus())


In [ ]:
# Component 4: Async Eval Pipeline + Regression Detector (condensed from L49)

JUDGE_TOOL = {
    'name': 'eval',
    'description': 'Evaluate the quality of an AI response',
    'input_schema': {
        'type': 'object',
        'required': ['score', 'verdict'],
        'properties': {
            'score':   {'type': 'number', 'description': '0.0 to 1.0'},
            'verdict': {'type': 'string', 'enum': ['good', 'acceptable', 'poor']}
        }
    }
}

@dataclass
class EvalSample:
    id: str; query: str; response: str; tag: str; latency_s: float


async def _judge(s: EvalSample) -> dict:
    r = await async_client.messages.create(
        model=HAIKU, max_tokens=100,
        tools=[JUDGE_TOOL], tool_choice={'type': 'tool', 'name': 'eval'},
        messages=[{'role': 'user', 'content':
            f'Query: {s.query}\n\nResponse: {s.response}\n\n'
            'Is this response accurate, relevant, and well-structured?'}])
    for b in r.content:
        if b.type == 'tool_use': return b.input
    return {'score': 0.5, 'verdict': 'acceptable'}


class EvalStore:
    def __init__(self):
        self.conn = sqlite3.connect(':memory:', check_same_thread=False)
        self.conn.execute(
            'CREATE TABLE scores (id TEXT PRIMARY KEY, ts REAL, tag TEXT, '
            'score REAL, verdict TEXT, latency_s REAL)')
        self.conn.commit()

    def write(self, s: EvalSample, r: dict):
        self.conn.execute(
            'INSERT OR REPLACE INTO scores VALUES (?,?,?,?,?,?)',
            (s.id, time.time(), s.tag, r['score'], r['verdict'], s.latency_s))
        self.conn.commit()

    def df(self) -> pd.DataFrame:
        return pd.read_sql('SELECT * FROM scores ORDER BY ts', self.conn)


class AsyncEvalPipeline:
    def __init__(self, store: EvalStore, metrics: ProdMetrics, base_rate=0.4):
        self.store = store; self.metrics = metrics; self.base_rate = base_rate
        self.queue: asyncio.Queue = asyncio.Queue(maxsize=300)
        self.judged = self.dropped = 0

    def enqueue(self, s: EvalSample):
        importance = 0.4 * min(s.latency_s / 10, 1.0) + \
                     0.6 * min(len(s.response) / 600, 1.0)
        effective_rate = min(self.base_rate + importance * 0.4, 1.0)
        if random.random() < effective_rate:
            try: self.queue.put_nowait(s)
            except asyncio.QueueFull: self.dropped += 1

    async def drain(self, limit=30):
        count = 0
        while count < limit and not self.queue.empty():
            s = await self.queue.get()
            try:
                result = await _judge(s)
                self.store.write(s, result)
                self.metrics.record_score(result['score'])
                self.judged += 1
            except Exception: pass
            count += 1


class RegressionDetector:
    def __init__(self, store: EvalStore): self.store = store

    def check(self, current_h=0.5, baseline_h=12, min_n=5) -> list:
        df = self.store.df()
        if df.empty or len(df) < min_n * 2: return []
        alerts = []
        cutoff = time.time() - current_h * 3600
        for tag in list(df['tag'].unique()) + ['__all__']:
            sub      = df if tag == '__all__' else df[df['tag'] == tag]
            current  = sub[sub['ts'] >= cutoff]['score']
            baseline = sub[sub['ts'] <  cutoff]['score']
            if len(current) < min_n or len(baseline) < min_n: continue
            _, p = mannwhitneyu(current, baseline, alternative='less')
            delta = current.mean() - baseline.mean()
            if p < 0.05 and delta < -0.05:
                sev = 'CRITICAL' if delta < -0.15 else 'WARNING'
                alerts.append(f'[{sev}] {tag}: score dropped {delta:+.3f} (p={p:.4f})')
        return alerts


EVAL_STORE = EvalStore()
EVAL_PIPE  = AsyncEvalPipeline(EVAL_STORE, METRICS)
DETECTOR   = RegressionDetector(EVAL_STORE)
print(f'Async eval + regression detector ready | base_rate={EVAL_PIPE.base_rate:.0%}')


## Wiring: `ProductionAIStack`

The stack is a thin coordinator -- it doesn't contain logic, just **sequence**:

```
infer(query, tag)
  |
  +-- 1. traced_infer()      <- durable_research() wrapped in OTel spans
  |         +--> DurableCtx.run() x3  (expand -> draft -> revise)
  |
  +-- 2. METRICS.record()    <- non-blocking, pure in-memory
  |
  +-- 3. EVAL_PIPE.enqueue() <- put_nowait() -- drops if queue full, never blocks
```

**Separately, on a timer (every 5 min in production):**
```
drain_evals()       <- async worker judges queued samples
check_regressions() <- Mann-Whitney U on current vs baseline window
```

**The key property:** `infer()` is synchronous and returns in O(LLM latency).
Eval and regression detection add **zero latency** to the hot path.


In [ ]:
# ProductionAIStack: Unified Entry Point

class ProductionAIStack:
    """
    Production AI inference stack -- wires all Track 5 components.

    Hot path (synchronous, runs on every request):
      infer() -> DurablePipeline -> OTelTracing -> Metrics -> EvalEnqueue

    Async background (call every 5 min):
      drain_evals()        -- judge sampled responses, write scores
      check_regressions()  -- Mann-Whitney U on recent vs baseline

    Exportable:
      .prometheus()   -> GET /metrics  -> Prometheus scrapes -> Grafana
      _span_exporter  -> OTel spans    -> Jaeger via OTLP
    """

    def __init__(self):
        self.metrics   = METRICS
        self.eval_pipe = EVAL_PIPE
        self.detector  = DETECTOR
        self.spans     = _span_exporter
        self._n        = 0

    def infer(self, query: str, tag: str = 'general') -> dict:
        self._n += 1
        wf_id = f'wf-{uuid.uuid4()}'
        t0    = time.perf_counter()
        try:
            result  = traced_infer(query, tag, wf_id)
            latency = time.perf_counter() - t0
            self.metrics.record(tag, latency, ok=True)
            sample = EvalSample(
                id=str(uuid.uuid4()), query=query,
                response=result['answer'], tag=tag, latency_s=latency)
            self.eval_pipe.enqueue(sample)
            return result
        except Exception as e:
            self.metrics.record(tag, time.perf_counter() - t0, ok=False)
            raise

    async def drain_evals(self, limit: int = 30):
        await self.eval_pipe.drain(limit=limit)

    def check_regressions(self) -> list:
        return self.detector.check()

    def status(self) -> dict:
        return {
            'requests':      self._n,
            'p95_latency_s': round(self.metrics.p95_latency, 3),
            'mean_score':    round(self.metrics.mean_score, 3),
            'poor_rate':     round(self.metrics.poor_rate, 3),
            'queue_depth':   self.eval_pipe.queue.qsize(),
            'judged':        self.eval_pipe.judged,
            'dropped':       self.eval_pipe.dropped,
            'otel_spans':    len(self.spans.get_finished_spans()),
        }

    def prometheus(self) -> str:
        return self.metrics.prometheus()


STACK = ProductionAIStack()
print('ProductionAIStack initialized')
print('  Hot path  : infer() -> Durable -> OTelTrace -> Metrics -> EvalEnqueue')
print('  Background: drain_evals() + check_regressions()')
print('  Export    : .prometheus() -> /metrics | spans -> Jaeger')


In [ ]:
# Live Demo: 15 requests through the full stack

DEMO_QUERIES = [
    ('What is gradient descent?',               'ml'),
    ('Explain the attention mechanism',          'nlp'),
    ('How does BERT differ from GPT?',           'nlp'),
    ('What is retrieval-augmented generation?',  'agents'),
    ('Explain LLM tool use',                    'agents'),
    ('What is RLHF?',                           'ml'),
    ('How does LoRA fine-tuning work?',          'ml'),
    ('What is vLLM continuous batching?',        'serving'),
    ('Explain OpenTelemetry spans',              'observability'),
    ('What are circuit breakers in ML systems?', 'observability'),
    ('How does the A2A protocol work?',          'agents'),
    ('What is knowledge distillation?',          'ml'),
    ('Explain model merging with SLERP',         'ml'),
    ('What is an async eval pipeline?',          'observability'),
    ('How does KV-cache work in transformers?',  'serving'),
]

print('Running 15 requests through ProductionAIStack...\n')
print(f'{"#":>3}  {"tag":<14}  {"latency":>7}  {"exec":>4}  {"rep":>3}  chars')
print('-' * 52)

for i, (q, tag) in enumerate(DEMO_QUERIES):
    r = STACK.infer(q, tag=tag)
    print(f'{i+1:3d}  {tag:<14}  {r["latency_s"]:>6.2f}s  '
          f'{r["executed"]:>4}  {r["replayed"]:>3}  {len(r["answer"])}')

s = STACK.status()
print(f'\n{"="*52}')
print(f'Done: {s["requests"]} requests | P95 latency: {s["p95_latency_s"]:.2f}s')
print(f'Eval queue: {s["queue_depth"]} pending | OTel spans: {s["otel_spans"]}')
print(f'\nSample /metrics output:')
print(STACK.prometheus()[:600])


In [ ]:
# Drain eval queue + check for regressions

print('Running async eval worker (judging sampled responses)...\n')
asyncio.run(STACK.drain_evals(limit=20))

s = STACK.status()
print(f'Post-eval status:')
print(f'  Judged     : {s["judged"]}')
print(f'  Dropped    : {s["dropped"]}')
print(f'  Mean score : {s["mean_score"]:.3f}')
print(f'  Poor rate  : {s["poor_rate"]:.1%}')

df = EVAL_STORE.df()
if not df.empty:
    print(f'\nEval results ({len(df)} judged responses):')
    print(df[['tag', 'score', 'verdict', 'latency_s']].to_string(index=False))
    print('\nMean score by tag:')
    for tag, grp in df.groupby('tag'):
        bar = chr(9608) * int(grp['score'].mean() * 20)
        print(f'  {tag:<14} {grp["score"].mean():.3f}  {bar}')

alerts = STACK.check_regressions()
if alerts:
    print(f'\nRegression alerts:')
    for a in alerts: print(f'  {a}')
else:
    print('\nNo regressions detected (need >= 5 samples in both windows)')
    print('In production: runs every 5 min against a 12h rolling baseline')

# EXPERIMENT: inject degraded scores to trigger a regression alert:
# for _ in range(10):
#     EVAL_STORE.write(
#         EvalSample(str(uuid.uuid4()), 'test', 'bad answer', 'ml', 1.0),
#         {'score': 0.2, 'verdict': 'poor'})
# print(STACK.check_regressions())


In [ ]:
# Production Dashboard (5-panel matplotlib)

df    = EVAL_STORE.df()
spans = _span_exporter.get_finished_spans()

fig = plt.figure(figsize=(16, 10))
fig.suptitle('ProductionAIStack -- Live Dashboard', fontsize=14, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.45)

# Panel 1: Requests by tag
ax1 = fig.add_subplot(gs[0, 0])
tags   = list(METRICS._req_count.keys())
counts = [METRICS._req_count[t] for t in tags]
bars   = ax1.barh(tags, counts, color='#4C72B0', edgecolor='white')
ax1.set_title('Requests by Tag', fontweight='bold')
ax1.set_xlabel('Count')
for b, c in zip(bars, counts):
    ax1.text(b.get_width() + 0.05, b.get_y() + b.get_height()/2,
             str(c), va='center', fontsize=9)

# Panel 2: Latency distribution
ax2 = fig.add_subplot(gs[0, 1])
n_bins = min(10, max(2, len(METRICS._latencies)//2))
ax2.hist(METRICS._latencies, bins=n_bins, color='#55A868', edgecolor='white', alpha=0.85)
p95 = METRICS.p95_latency
ax2.axvline(p95, color='red', linestyle='--', lw=1.5, label=f'P95={p95:.2f}s')
ax2.set_title('Latency Distribution', fontweight='bold')
ax2.set_xlabel('Seconds'); ax2.set_ylabel('Requests')
ax2.legend(fontsize=9)

# Panel 3: Mean eval score by tag
ax3 = fig.add_subplot(gs[0, 2])
if not df.empty:
    by_tag = df.groupby('tag')['score'].mean().sort_values()
    colors = ['#d62728' if s < 0.6 else '#2ca02c' for s in by_tag.values]
    ax3.barh(by_tag.index, by_tag.values, color=colors)
    ax3.axvline(0.6, color='orange', linestyle='--', lw=1.5, label='Threshold 0.6')
    ax3.set_xlim(0, 1.05); ax3.legend(fontsize=9)
else:
    ax3.text(0.5, 0.5, 'No eval data yet', ha='center', va='center',
             transform=ax3.transAxes, color='gray')
ax3.set_title('Mean Eval Score by Tag', fontweight='bold')
ax3.set_xlabel('Score (0-1)')

# Panel 4: OTel span waterfall (last 10 root spans)
ax4 = fig.add_subplot(gs[1, :2])
root_spans = [s for s in spans if s.name == 'pipeline.request'][-10:]
if root_spans:
    min_ts = min(s.start_time for s in root_spans) / 1e9
    cmap   = {'pipeline.request': '#4C72B0', 'pipeline.durable': '#55A868'}
    for i, span in enumerate(root_spans):
        start = span.start_time / 1e9 - min_ts
        dur   = (span.end_time - span.start_time) / 1e9
        ax4.barh(i, dur, left=start, color=cmap['pipeline.request'], alpha=0.5, height=0.5)
        children = [s for s in spans
                    if s.name == 'pipeline.durable'
                    and s.start_time >= span.start_time
                    and s.end_time   <= span.end_time + 1_000_000_000]
        for c in children:
            cs = c.start_time / 1e9 - min_ts
            cd = (c.end_time - c.start_time) / 1e9
            ax4.barh(i, cd, left=cs, color=cmap['pipeline.durable'], alpha=0.9, height=0.3)
    ax4.set_yticks(range(len(root_spans)))
    ax4.set_yticklabels([f'req-{i+1}' for i in range(len(root_spans))], fontsize=8)
    ax4.set_xlabel('Time from first request (s)')
    handles = [plt.Rectangle((0,0),1,1,color=c,alpha=0.8,label=n) for n,c in cmap.items()]
    ax4.legend(handles=handles, fontsize=9, loc='lower right')
ax4.set_title('OTel Span Timeline (last 10 root spans)', fontweight='bold')

# Panel 5: Verdict distribution
ax5 = fig.add_subplot(gs[1, 2])
if not df.empty and len(df) >= 2:
    vc     = df['verdict'].value_counts()
    cmap_v = {'good': '#2ca02c', 'acceptable': '#ff7f0e', 'poor': '#d62728'}
    ax5.pie(vc.values, labels=vc.index,
            colors=[cmap_v.get(v, 'gray') for v in vc.index],
            autopct='%1.0f%%', startangle=90)
else:
    ax5.text(0.5, 0.5, 'Insufficient eval data\n(need >= 2 judged responses)',
             ha='center', va='center', transform=ax5.transAxes, color='gray', fontsize=9)
ax5.set_title('Eval Verdict Distribution', fontweight='bold')

plt.savefig('/content/prod_dashboard.png', dpi=110, bbox_inches='tight')
plt.show()
print('Dashboard saved to /content/prod_dashboard.png')


## Ship Artifacts -- Production Deployment

The cell below writes out a complete **`ai-stack/`** directory you can push to GitHub
and deploy with `docker compose up -d`.

```
ai-stack/
  app/
    Dockerfile
    main.py          <- FastAPI app wrapping ProductionAIStack
  grafana/
    provisioning/
      datasources/prometheus.yml
      dashboards/dashboard.yml
    dashboards/
      ai_stack.json  <- Pre-built Grafana dashboard (6 panels)
  prometheus.yml
  docker-compose.yml
```

**After `docker compose up -d`:**

| Service | URL | What you see |
|---------|-----|-------------|
| AI App | http://localhost:8000/docs | FastAPI Swagger UI |
| Jaeger | http://localhost:16686 | Trace explorer |
| Prometheus | http://localhost:9090 | Metrics query UI |
| Grafana | http://localhost:3000 | Live dashboard (admin/admin) |


In [ ]:
# Generate ai-stack/ production deployment skeleton
import pathlib

BASE = pathlib.Path('/content/ai-stack')
(BASE / 'app').mkdir(parents=True, exist_ok=True)
(BASE / 'grafana/provisioning/datasources').mkdir(parents=True, exist_ok=True)
(BASE / 'grafana/provisioning/dashboards').mkdir(parents=True, exist_ok=True)
(BASE / 'grafana/dashboards').mkdir(parents=True, exist_ok=True)

DOCKER_COMPOSE = '''\
version: "3.9"

services:
  ai-app:
    build: ./app
    ports: ["8000:8000"]
    environment:
      - ANTHROPIC_API_KEY=${ANTHROPIC_API_KEY}
      - OTEL_SERVICE_NAME=ai-production-stack
      - OTEL_EXPORTER_OTLP_ENDPOINT=http://jaeger:4317
    depends_on: [jaeger, prometheus]
    restart: unless-stopped

  jaeger:
    image: jaegertracing/all-in-one:1.57
    ports:
      - "16686:16686"
      - "4317:4317"
      - "4318:4318"
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:v2.51.2
    ports: ["9090:9090"]
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    restart: unless-stopped

  grafana:
    image: grafana/grafana:10.4.2
    ports: ["3000:3000"]
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=admin
      - GF_USERS_ALLOW_SIGN_UP=false
    volumes:
      - ./grafana/provisioning:/etc/grafana/provisioning
      - ./grafana/dashboards:/var/lib/grafana/dashboards
    depends_on: [prometheus, jaeger]
    restart: unless-stopped
'''

PROMETHEUS_YML = '''\
global:
  scrape_interval: 15s

scrape_configs:
  - job_name: ai-stack
    static_configs:
      - targets: ["ai-app:8000"]
    metrics_path: /metrics
'''

DOCKERFILE = '''\
FROM python:3.11-slim
WORKDIR /app
RUN pip install --no-cache-dir fastapi uvicorn anthropic opentelemetry-api \\
    opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc scipy pandas
COPY main.py .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

MAIN_PY = '''\
"""FastAPI wrapper around ProductionAIStack."""
import asyncio
from fastapi import FastAPI
from fastapi.responses import PlainTextResponse
from pydantic import BaseModel
# from stack import STACK  # wire in your ProductionAIStack instance

app = FastAPI(title="AI Production Stack", version="1.0.0")

class InferRequest(BaseModel):
    query: str
    tag: str = "general"

@app.post("/infer")
def infer(req: InferRequest):
    result = STACK.infer(req.query, tag=req.tag)
    return {"answer": result["answer"], "latency_s": result["latency_s"]}

@app.get("/metrics", response_class=PlainTextResponse)
def metrics():
    return STACK.prometheus()

@app.get("/status")
def status():
    return STACK.status()

@app.on_event("startup")
async def start_eval_worker():
    async def _loop():
        while True:
            await asyncio.sleep(300)
            await STACK.drain_evals(limit=50)
    asyncio.create_task(_loop())
'''

DS_YML = '''\
apiVersion: 1
datasources:
  - name: Prometheus
    type: prometheus
    url: http://prometheus:9090
    isDefault: true
  - name: Jaeger
    type: jaeger
    url: http://jaeger:16686
'''

DB_YML = '''\
apiVersion: 1
providers:
  - name: default
    folder: AI Stack
    type: file
    options:
      path: /var/lib/grafana/dashboards
'''

(BASE / 'docker-compose.yml').write_text(DOCKER_COMPOSE)
(BASE / 'prometheus.yml').write_text(PROMETHEUS_YML)
(BASE / 'app/Dockerfile').write_text(DOCKERFILE)
(BASE / 'app/main.py').write_text(MAIN_PY)
(BASE / 'grafana/provisioning/datasources/prometheus.yml').write_text(DS_YML)
(BASE / 'grafana/provisioning/dashboards/dashboard.yml').write_text(DB_YML)

dashboard = {
    'title': 'AI Production Stack', 'uid': 'ai-prod-stack', 'refresh': '30s',
    'time': {'from': 'now-1h', 'to': 'now'},
    'panels': [
        {'id':1,'type':'stat','title':'Total Requests',
         'targets':[{'expr':'sum(ai_requests_total)'}],
         'gridPos':{'x':0,'y':0,'w':6,'h':4}},
        {'id':2,'type':'gauge','title':'P95 Latency (s)',
         'targets':[{'expr':'ai_latency_p95_seconds'}],
         'fieldConfig':{'defaults':{'thresholds':{'steps':[
             {'color':'green','value':0},{'color':'orange','value':3},{'color':'red','value':8}]}}},
         'gridPos':{'x':6,'y':0,'w':6,'h':4}},
        {'id':3,'type':'gauge','title':'Mean Eval Score',
         'targets':[{'expr':'ai_eval_mean_score'}],
         'fieldConfig':{'defaults':{'min':0,'max':1,'thresholds':{'steps':[
             {'color':'red','value':0},{'color':'orange','value':0.6},{'color':'green','value':0.8}]}}},
         'gridPos':{'x':12,'y':0,'w':6,'h':4}},
        {'id':4,'type':'gauge','title':'Poor Rate',
         'targets':[{'expr':'ai_eval_poor_rate'}],
         'fieldConfig':{'defaults':{'min':0,'max':1,'thresholds':{'steps':[
             {'color':'green','value':0},{'color':'orange','value':0.1},{'color':'red','value':0.2}]}}},
         'gridPos':{'x':18,'y':0,'w':6,'h':4}},
        {'id':5,'type':'timeseries','title':'Request Rate by Tag (req/min)',
         'targets':[{'expr':'rate(ai_requests_total[1m])*60','legendFormat':'{{tag}}'}],
         'gridPos':{'x':0,'y':4,'w':12,'h':8}},
        {'id':6,'type':'timeseries','title':'Eval Quality Over Time',
         'targets':[
             {'expr':'ai_eval_mean_score','legendFormat':'mean score'},
             {'expr':'ai_eval_poor_rate','legendFormat':'poor rate'}],
         'gridPos':{'x':12,'y':4,'w':12,'h':8}},
    ]
}
(BASE / 'grafana/dashboards/ai_stack.json').write_text(json.dumps(dashboard, indent=2))

print('ai-stack/ deployment artifacts written:\n')
for p in sorted(BASE.rglob('*')):
    depth  = len(p.relative_to(BASE).parts) - 1
    indent = '  ' * depth
    icon   = '[DIR] ' if p.is_dir() else '[FILE]'
    print(f'{indent}{icon} {p.name}')

print('\nTo deploy:')
print('  cd /content/ai-stack')
print('  ANTHROPIC_API_KEY=sk-... docker compose up -d')
print('  open http://localhost:3000   # Grafana (admin/admin)')
print('  open http://localhost:16686  # Jaeger traces')


## 10 Capstone Pitfalls

| # | Pitfall | Why it burns you | Fix |
|---|---------|-----------------|-----|
| 1 | **Eval on hot path** | Every request waits for Haiku judge -> 2x latency | `queue.put_nowait()` -- drop if full, never block |
| 2 | **Single SQLite writer** | Concurrent threads deadlock on `conn.commit()` | `check_same_thread=False` + WAL mode, or use Postgres |
| 3 | **InMemorySpanExporter in production** | Spans accumulate -> OOM after hours | Swap for `OTLPSpanExporter` + `BatchSpanProcessor` |
| 4 | **Prometheus gauge vs counter** | `mean_score` set as counter -> always rising | Scores are gauges (can decrease); request counts are counters |
| 5 | **Regression with N < 10** | Mann-Whitney U p-values are random with tiny samples | Enforce `min_n` guard; log 'insufficient data' not an alert |
| 6 | **No base_rate floor for eval** | Importance sampling can zero out fast, short responses | Always sample errors; set `base_rate >= 0.2` |
| 7 | **In-memory WorkflowDB per replica** | Multiple replicas each use `:memory:` -- no shared durability | Use a shared file path or Redis/Postgres |
| 8 | **Grafana alert on poor_rate > 0.05** | Too sensitive -- normal variance triggers false alarms | Add `AND count() > 50` guard in Prometheus query |
| 9 | **Missing `restart: unless-stopped`** | One container crash stops the whole stack silently | Health checks + restart policy in every service |
| 10 | **Wiring OTel after first request** | First requests are un-traced, breaks span count assumptions | Set `TracerProvider` at import time, before any LLM call |

**The capstone rule:** _Observability is not a feature you add later. Wire it at import time._


## Track 5 COMPLETE

| Track | Lessons | What you can now build |
|-------|---------|----------------------|
| Track 1 | L24-L31 | Reliability harness (circuit breakers, canary, A/B, SLOs) |
| Track 2 | L32-L36 | Multi-agent swarms (A2A, Blackboard, Debate, Fan-out) |
| Track 3 | L37-L41 | Self-hosted LLMs (vLLM, QLoRA, DPO, distillation, merging) |
| Track 4 | L42-L45 | Multimodal agents (Voice, Image, Document AI) |
| **Track 5** | **L46-L50** | **Agent-ops (Durable, Autoscale, OTel, Eval, Deploy)** |

## 5 Homework Tasks

1. **Wire a real OTLP exporter** -- swap `InMemorySpanExporter` for `OTLPSpanExporter('http://localhost:4317')`, run Jaeger locally with `docker run -p 16686:16686 -p 4317:4317 jaegertracing/all-in-one`, and confirm you see spans in the Jaeger UI.

2. **Add a real Prometheus histogram** -- replace the `_latencies` list in `ProdMetrics` with an `opentelemetry.metrics.Histogram('ai.latency.seconds')`. Verify the P95 changes as requests flow in.

3. **Test multi-replica durability** -- change `WorkflowDB(':memory:')` to `WorkflowDB('/tmp/wf.db')`, crash the kernel mid-pipeline by inserting `raise RuntimeError('crash!')` after the `expand` step, then restart and re-run the same `wf_id`. Verify only the remaining steps execute.

4. **Add a Grafana alert rule** -- in the Grafana UI, add an alert on `ai_eval_poor_rate > 0.2 for 10m` that sends a Slack webhook. Document the YAML.

5. **Open-source the stack** -- push `ai-stack/` to GitHub, wire the docker-compose into a GitHub Actions workflow that builds and runs a smoke test (`curl http://localhost:8000/status`). This is your public proof of work as an AI engineer.

## What's Next -- Phase 5

You have now completed **all 5 Tracks** of Phase 4. The full arc:

```
Phase 1  -> LLM API fundamentals, prompt engineering, first agents
Phase 2  -> Structured outputs, fine-tuning, MCP, production basics
Phase 3  -> FastAPI deployment, advanced evals, AI security, streaming
Phase 4  -> Reliability, multi-agent, self-hosted, multimodal, agent-ops
```

**Phase 5 proposal: Build Your Open-Source Project**

Take the `auto_researcher` skeleton from L31/L36, then:
- Wire in Track 3 (self-hosted model via vLLM)
- Add Track 4 multimodal input (PDF + voice query)
- Wrap with full Track 5 observability (OTel + eval + Grafana)
- Ship to PyPI with a live Grafana dashboard screenshot in the README

That is the portfolio piece that proves you are a production AI engineer.
We will design the Phase 5 curriculum together on the next run.
